# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# ============================================================
# ML-08 — SETUP
# ============================================================

import duckdb
import pandas as pd
import numpy as np
import os

# ------------------------------------------------------------
# 1. Create DuckDB connection
# ------------------------------------------------------------

con = duckdb.connect()

# ------------------------------------------------------------
# 2. Hugging Face warehouse
# ------------------------------------------------------------

warehouse = "hf://datasets/FlyRank/internship-warehouse"

# ------------------------------------------------------------
# 3. Hugging Face authentication
# ------------------------------------------------------------

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN not found. Add your Hugging Face token "
        "to Colab Secrets as HF_TOKEN."
    )

# ------------------------------------------------------------
# 4. Create Hugging Face secret
# ------------------------------------------------------------

con.execute("DROP SECRET IF EXISTS hf_secret")

con.execute(f"""
    CREATE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
""")

# ------------------------------------------------------------
# 5. Performance data path
# ------------------------------------------------------------

PERF_GLOB = (
    f"{warehouse}/fact_content_daily_performance/"
    "month=*/data_0.parquet"
)

print("DuckDB connection: OK")
print("Hugging Face authentication: OK")
print("Performance path configured.")

DuckDB connection: OK
Hugging Face authentication: OK
Performance path configured.


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I use a Random Forest Regressor to estimate future content CTR from earlier observed performance signals. This method fits the lane because it can capture non-linear relationships between impressions, clicks, position, traffic, and engagement without requiring a linear relationship. It also gives feature importance measures that can support interpretation.

The model is used for decision-support rather than as proof that a content change will improve performance.


In [2]:
# ============================================================
# ML-08 SECTION 1: DATA SOURCE CHECK
# ============================================================

source_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet(
        '{PERF_GLOB}',
        hive_partitioning=true
    )
    WHERE report_date >= '2026-01-01'
      AND report_date < '2026-07-01'
""").fetchdf()

display(source_check)

print("Section 1 data check completed.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,min_date,max_date
0,58893481,2026-01-01,2026-06-30


Section 1 data check completed.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a time-aware split because the goal is to estimate future search performance from information that would have been available earlier. January–April 2026 is used for training, May 2026 is used for validation, and June 2026 is held out as the final test period. This avoids using future observations to predict earlier observations and is more realistic than a random split.


In [3]:
# ============================================================
# ML-08 SECTION 2: TIME-AWARE SPLIT
# ============================================================

# ------------------------------------------------------------
# 1. Check data coverage
# ------------------------------------------------------------

coverage = con.execute(f"""
    SELECT
        COUNT(*) AS rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet(
        '{PERF_GLOB}',
        hive_partitioning=true
    )
    WHERE report_date >= '2026-01-01'
      AND report_date < '2026-07-01'
      AND gsc_impressions IS NOT NULL
      AND gsc_clicks IS NOT NULL
      AND gsc_impressions > 0
""").fetchdf()

print("Data coverage:")
display(coverage)

# ------------------------------------------------------------
# 2. Count rows in each time split
# ------------------------------------------------------------

split_summary = con.execute(f"""
    SELECT
        CASE
            WHEN report_date >= '2026-01-01'
             AND report_date < '2026-05-01'
                THEN 'train'

            WHEN report_date >= '2026-05-01'
             AND report_date < '2026-06-01'
                THEN 'valid'

            WHEN report_date >= '2026-06-01'
             AND report_date < '2026-07-01'
                THEN 'test'
        END AS split,

        COUNT(*) AS rows,

        MIN(report_date) AS min_date,

        MAX(report_date) AS max_date

    FROM read_parquet(
        '{PERF_GLOB}',
        hive_partitioning=true
    )

    WHERE report_date >= '2026-01-01'
      AND report_date < '2026-07-01'
      AND gsc_impressions IS NOT NULL
      AND gsc_clicks IS NOT NULL
      AND gsc_impressions > 0

    GROUP BY split

    ORDER BY
        CASE split
            WHEN 'train' THEN 1
            WHEN 'valid' THEN 2
            WHEN 'test' THEN 3
        END
""").fetchdf()

print("\nTime-aware split:")
display(split_summary)

# ------------------------------------------------------------
# 3. Validate chronological order
# ------------------------------------------------------------

assert split_summary["split"].tolist() == [
    "train",
    "valid",
    "test"
]

assert (
    split_summary.loc[0, "max_date"]
    < split_summary.loc[1, "min_date"]
)

assert (
    split_summary.loc[1, "max_date"]
    < split_summary.loc[2, "min_date"]
)

print("\nChronological split check: PASSED")
print("Section 2 completed successfully.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data coverage:


,rows,min_date,max_date
0,20783406,2026-01-01,2026-06-30


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Time-aware split:


,split,rows,min_date,max_date
0,train,12531047,2026-01-01,2026-04-30
1,valid,4373422,2026-05-01,2026-05-31
2,test,3878937,2026-06-01,2026-06-30



Chronological split check: PASSED
Section 2 completed successfully.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I compare the Random Forest with a simple historical-CTR baseline using the same time-aware validation and test periods. The model predicts future content CTR from historical search and engagement signals. I report MAE because it is directly interpretable as average absolute CTR prediction error. Results are treated as decision-support rather than proof of causal improvement.

In [4]:
# ============================================================
# ML-08 SECTION 3: TRAIN + COMPARE WITH BASELINE
# RAM-SAFE VERSION
# ============================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Build content-level historical features in DuckDB
# ------------------------------------------------------------
#
# Training features:
#   Jan-Apr historical performance
#
# Target:
#   May CTR
#
# Test:
#   Train/validation development -> June CTR
#
# ------------------------------------------------------------

model_df = con.execute(f"""
WITH base AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        CAST(gsc_impressions AS DOUBLE) AS impressions,
        CAST(gsc_clicks AS DOUBLE) AS clicks,
        CAST(gsc_avg_position AS DOUBLE) AS avg_position,
        CAST(ga4_pageviews AS DOUBLE) AS pageviews,
        CAST(ga4_sessions AS DOUBLE) AS sessions,
        CAST(ga4_users AS DOUBLE) AS users,
        CAST(ga4_engaged_sessions AS DOUBLE) AS engaged_sessions,
        CAST(scroll_events AS DOUBLE) AS scroll_events

    FROM read_parquet(
        '{PERF_GLOB}',
        hive_partitioning=true
    )

    WHERE report_date >= '2026-01-01'
      AND report_date < '2026-07-01'
      AND gsc_impressions IS NOT NULL
      AND gsc_clicks IS NOT NULL
      AND gsc_impressions > 0
),

historical AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(impressions) AS hist_impressions,
        SUM(clicks) AS hist_clicks,

        AVG(
            CASE
                WHEN impressions > 0
                THEN clicks / impressions
            END
        ) AS hist_ctr,

        AVG(avg_position) AS hist_avg_position,

        SUM(pageviews) AS hist_pageviews,
        SUM(sessions) AS hist_sessions,
        SUM(users) AS hist_users,
        SUM(engaged_sessions) AS hist_engaged_sessions,
        SUM(scroll_events) AS hist_scroll_events

    FROM base

    WHERE report_date >= '2026-01-01'
      AND report_date < '2026-05-01'

    GROUP BY
        client_hash_id,
        content_hash_id
),

may_target AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(impressions) AS may_impressions,
        SUM(clicks) AS may_clicks,

        CASE
            WHEN SUM(impressions) > 0
            THEN SUM(clicks) / SUM(impressions)
        END AS may_ctr

    FROM base

    WHERE report_date >= '2026-05-01'
      AND report_date < '2026-06-01'

    GROUP BY
        client_hash_id,
        content_hash_id
),

june_target AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(impressions) AS june_impressions,
        SUM(clicks) AS june_clicks,

        CASE
            WHEN SUM(impressions) > 0
            THEN SUM(clicks) / SUM(impressions)
        END AS june_ctr

    FROM base

    WHERE report_date >= '2026-06-01'
      AND report_date < '2026-07-01'

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    h.*,

    m.may_impressions,
    m.may_clicks,
    m.may_ctr,

    j.june_impressions,
    j.june_clicks,
    j.june_ctr

FROM historical h

INNER JOIN may_target m
    ON h.client_hash_id = m.client_hash_id
   AND h.content_hash_id = m.content_hash_id

INNER JOIN june_target j
    ON h.client_hash_id = j.client_hash_id
   AND h.content_hash_id = j.content_hash_id

WHERE m.may_ctr IS NOT NULL
  AND j.june_ctr IS NOT NULL
""").fetchdf()

print("Modeling rows:", len(model_df))
print("Modeling columns:", len(model_df.columns))

display(model_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling rows: 160595
Modeling columns: 17


,client_hash_id,content_hash_id,hist_impressions,hist_clicks,hist_ctr,hist_avg_position,hist_pageviews,hist_sessions,hist_users,hist_engaged_sessions,hist_scroll_events,may_impressions,may_clicks,may_ctr,june_impressions,june_clicks,june_ctr
0,client_e547b89c05043229,content_cc02e6a397d2ffc2,8740.0,36.0,0.004105,5.681472,31.0,24.0,24.0,0.0,9.0,1545.0,6.0,0.003883,2258.0,8.0,0.003543
1,client_e547b89c05043229,content_52d68cbbd7862053,4016.0,10.0,0.003097,12.422197,10.0,9.0,9.0,2.0,3.0,1310.0,8.0,0.006107,5800.0,14.0,0.002414
2,client_e547b89c05043229,content_782d0c9cd351dcb4,3184.0,16.0,0.004918,14.646350,26.0,22.0,22.0,2.0,4.0,2383.0,17.0,0.007134,8274.0,105.0,0.012690
3,client_e547b89c05043229,content_7916e8f527f1c672,1310.0,2.0,0.001499,29.415953,3.0,2.0,2.0,0.0,0.0,343.0,0.0,0.000000,348.0,0.0,0.000000
4,client_e547b89c05043229,content_e79bcd3d80e36f91,2262.0,1.0,0.000575,27.098766,3.0,3.0,3.0,0.0,0.0,482.0,2.0,0.004149,423.0,0.0,0.000000


In [5]:
# ============================================================
# 2. Feature / target definition
# ============================================================

features = [
    "hist_impressions",
    "hist_clicks",
    "hist_ctr",
    "hist_avg_position",
    "hist_pageviews",
    "hist_sessions",
    "hist_users",
    "hist_engaged_sessions",
    "hist_scroll_events",
]

X = model_df[features].replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

y_may = model_df["may_ctr"]
y_june = model_df["june_ctr"]

# ------------------------------------------------------------
# 3. Time-aware development split
# ------------------------------------------------------------

# Use May as validation target.
# June remains the final test target.

X_train = X
y_train = y_may

# ------------------------------------------------------------
# 4. Historical CTR baseline
# ------------------------------------------------------------

baseline_may = X_train["hist_ctr"]

baseline_may_mae = mean_absolute_error(
    y_may,
    baseline_may
)

print(
    "Baseline validation MAE:",
    baseline_may_mae
)

# ------------------------------------------------------------
# 5. Random Forest
# ------------------------------------------------------------

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

rf.fit(
    X_train,
    y_may
)

rf_may_pred = rf.predict(X_train)

rf_may_mae = mean_absolute_error(
    y_may,
    rf_may_pred
)

print(
    "Random Forest validation MAE:",
    rf_may_mae
)

Baseline validation MAE: 0.004509055981704071
Random Forest validation MAE: 0.0034727500300120416


In [7]:
# ============================================================
# ML-08 SECTION 3: TRAIN + COMPARE VS BASELINE
# ============================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Feature columns
# ------------------------------------------------------------

features = [
    "hist_impressions",
    "hist_clicks",
    "hist_ctr",
    "hist_avg_position",
    "hist_pageviews",
    "hist_sessions",
    "hist_users",
    "hist_engaged_sessions",
    "hist_scroll_events",
]

X = (
    model_df[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y_may = model_df["may_ctr"].astype(float)
y_june = model_df["june_ctr"].astype(float)

# ------------------------------------------------------------
# 2. Train Random Forest on January-April features
# ------------------------------------------------------------

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

rf.fit(X, y_may)

# ------------------------------------------------------------
# 3. May validation predictions
# ------------------------------------------------------------

rf_may_pred = rf.predict(X)

# Historical CTR baseline
baseline_may_pred = X["hist_ctr"].values

baseline_may_mae = mean_absolute_error(
    y_may,
    baseline_may_pred
)

rf_may_mae = mean_absolute_error(
    y_may,
    rf_may_pred
)

# ------------------------------------------------------------
# 4. June test predictions
# ------------------------------------------------------------

rf_june_pred = rf.predict(X)

baseline_june_pred = X["hist_ctr"].values

baseline_june_mae = mean_absolute_error(
    y_june,
    baseline_june_pred
)

rf_june_mae = mean_absolute_error(
    y_june,
    rf_june_pred
)

# ------------------------------------------------------------
# 5. Comparison table
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "model": [
        "Historical CTR baseline",
        "Random Forest"
    ],

    "validation_mae": [
        baseline_may_mae,
        rf_may_mae
    ],

    "test_mae": [
        baseline_june_mae,
        rf_june_mae
    ]
})

display(comparison)

# ------------------------------------------------------------
# 6. Improvement vs baseline
# ------------------------------------------------------------

test_improvement = (
    (baseline_june_mae - rf_june_mae)
    / baseline_june_mae
) * 100

print(
    f"Test MAE improvement vs baseline: "
    f"{test_improvement:.2f}%"
)

# ------------------------------------------------------------
# 7. Basic checks
# ------------------------------------------------------------

assert np.isfinite(comparison["validation_mae"]).all()
assert np.isfinite(comparison["test_mae"]).all()

print("Section 3 completed successfully.")

,model,validation_mae,test_mae
0,Historical CTR baseline,0.004509,0.005710
1,Random Forest,0.003473,0.004759


Test MAE improvement vs baseline: 16.65%
Section 3 completed successfully.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The Random Forest had lower error than the historical-CTR baseline on the held-out June period. I inspect the largest prediction errors and feature importance to understand where the model is less reliable and which historical signals it relies on. These results are directional and intended for decision-support rather than causal interpretation.

In [8]:
# ============================================================
# ML-08 SECTION 4: ERRORS AND INTERPRETATION
# ============================================================

# ------------------------------------------------------------
# 1. Create test-error table
# ------------------------------------------------------------

error_df = model_df[
    [
        "client_hash_id",
        "content_hash_id",
        "june_impressions",
        "june_clicks",
        "june_ctr"
    ]
].copy()

error_df["predicted_ctr"] = rf_june_pred

error_df["absolute_error"] = (
    error_df["june_ctr"] -
    error_df["predicted_ctr"]
).abs()

error_df["signed_error"] = (
    error_df["june_ctr"] -
    error_df["predicted_ctr"]
)

# ------------------------------------------------------------
# 2. Largest errors
# ------------------------------------------------------------

largest_errors = (
    error_df
    .sort_values("absolute_error", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

print("Top 20 largest June prediction errors:")
display(largest_errors)

# ------------------------------------------------------------
# 3. Error summary
# ------------------------------------------------------------

print("\nError summary:")
print(
    error_df["absolute_error"].describe()
)

# ------------------------------------------------------------
# 4. Feature importance
# ------------------------------------------------------------

feature_importance = pd.DataFrame({
    "feature": features,
    "importance": rf.feature_importances_
})

feature_importance = (
    feature_importance
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("\nRandom Forest feature importance:")
display(feature_importance)

# ------------------------------------------------------------
# 5. Most common error direction
# ------------------------------------------------------------

overprediction_rate = (
    error_df["predicted_ctr"] >
    error_df["june_ctr"]
).mean()

underprediction_rate = (
    error_df["predicted_ctr"] <
    error_df["june_ctr"]
).mean()

print(
    f"\nOverprediction rate: "
    f"{overprediction_rate:.2%}"
)

print(
    f"Underprediction rate: "
    f"{underprediction_rate:.2%}"
)

# ------------------------------------------------------------
# 6. Final checks
# ------------------------------------------------------------

assert len(largest_errors) > 0
assert len(feature_importance) == len(features)

print("\nSection 4 completed successfully.")

Top 20 largest June prediction errors:


,client_hash_id,content_hash_id,june_impressions,june_clicks,june_ctr,predicted_ctr,absolute_error,signed_error
0,client_08a6a72ff48e62c0,content_8155555258061027,1.0,1.0,1.0,0.000136,0.999864,0.999864
1,client_2094c6eb080311d5,content_8ffa40e4ad8900e6,1.0,1.0,1.0,0.000598,0.999402,0.999402
2,client_08a6a72ff48e62c0,content_d5e261a247448aba,1.0,1.0,1.0,0.000955,0.999045,0.999045
3,client_f623b01661d4bfe4,content_856fd3712cba07fa,1.0,1.0,1.0,0.000976,0.999024,0.999024
4,client_f623b01661d4bfe4,content_69b256e8a14a8591,1.0,1.0,1.0,0.000984,0.999016,0.999016
5,client_f623b01661d4bfe4,content_5ee3843d2b5490e0,1.0,1.0,1.0,0.001042,0.998958,0.998958
6,client_fef1a8f436438636,content_f7305884f08e1d4e,1.0,1.0,1.0,0.001100,0.998900,0.998900
7,client_23a62021009f63c4,content_97399540b42c4469,1.0,1.0,1.0,0.001444,0.998556,0.998556
8,client_2094c6eb080311d5,content_4c25a1e2c580e6fa,1.0,1.0,1.0,0.001540,0.998460,0.998460
9,client_08a6a72ff48e62c0,content_9f8aea4b32bdc4a6,1.0,1.0,1.0,0.001786,0.998214,0.998214



Error summary:
count    1.605950e+05
mean     4.758867e-03
std      2.784469e-02
min      2.829998e-08
25%      1.150405e-03
50%      2.059070e-03
75%      3.572368e-03
max      9.998641e-01
Name: absolute_error, dtype: float64

Random Forest feature importance:


,feature,importance
0,hist_ctr,0.368633
1,hist_avg_position,0.334657
2,hist_impressions,0.162796
3,hist_clicks,0.069119
4,hist_pageviews,0.024014
5,hist_scroll_events,0.018415
6,hist_sessions,0.010336
7,hist_users,0.007728
8,hist_engaged_sessions,0.004303



Overprediction rate: 74.02%
Underprediction rate: 25.98%

Section 4 completed successfully.


The largest prediction errors were concentrated in very low-volume observations where June had only one impression and one click, producing a CTR of 1.0. The model predicted much lower CTR values for these observations, so these cases dominate the maximum error. Across all 160,595 observations, the mean absolute error was 0.004759, with a median absolute error of 0.002059. The Random Forest relied most on historical CTR (0.368633) and historical average position (0.334657), followed by historical impressions (0.162796) and historical clicks (0.069119). The model overpredicted the observed June CTR in 74.02% of cases and underpredicted it in 25.98%. These results suggest that historical search performance is the strongest observed signal in this model, while very low-volume observations remain difficult to predict reliably.

## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.